# Calibration QA results

Displays the output of `run_bias_qa.py` (and later `run_flat_qa.py` / `run_fringe_qa.py`):
the per-(detector, amp, metric) table, the summary JSON, the per-exposure trend table and the
figures. Set `OUT` to the `--out` directory of a run and `CAL_TYPE` to its calibration type.

Thresholds in the default `calqa_config.yaml` are **placeholders** until a `--build-baseline`
config is used; check `summary["run"]["placeholder_thresholds"]` below.

In [ ]:
import os
OUT = os.path.expandvars("$LOGDIR/calibQa/bias/certified")  # the run's --out directory
CAL_TYPE = "bias"  # bias, flat or fringe
METRIC = "read_noise_e"  # metric for the detector x amp table below

In [ ]:
%matplotlib inline
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

sys.path.insert(0, os.getcwd())  # verify/calibs, for `common`
from common.plots import SEQUENTIAL

out = Path(OUT)
df = pd.read_parquet(out / f"{CAL_TYPE}_qa.parquet")
summary = json.loads((out / f"{CAL_TYPE}_qa_summary.json").read_text())
trend_path = out / f"{CAL_TYPE}_qa_trend.parquet"
trend = pd.read_parquet(trend_path) if trend_path.exists() else None
print(f"{len(df)} rows, {df['detector'].nunique()} detectors, {df['calib_id'].nunique()} calibs")
summary["run"]

## Flag counts per metric

In [ ]:
print("Totals:", summary["totals"], "| rows with errors:", summary["n_errors"])
pd.DataFrame(summary["metrics"]).T

## Warnings and failures

In [ ]:
flagged = df[df["flag"].isin(["warn", "fail"])]
cols = ["validity_begin", "filter", "detector", "amp", "metric", "value", "unit", "threshold", "flag"]
flagged.sort_values(["flag", "metric", "detector", "amp"])[cols]

In [ ]:
# rows that could not be measured, or carry a note
df[df["error"].fillna("") != ""][["validity_begin", "detector", "amp", "metric", "error"]]

## One metric per detector and amp

One column per (validity period, amp); colour is the metric value.

In [ ]:
table = df[df["metric"] == METRIC].pivot_table(
    index="detector", columns=["validity_begin", "amp"], values="value")
table.columns = [f"{str(v)[:10]} {a}" for v, a in table.columns]
fig, ax = plt.subplots(figsize=(1 + 0.6 * table.shape[1], 0.18 * table.shape[0] + 1))
im = ax.imshow(table.to_numpy(), aspect="auto", cmap=SEQUENTIAL, interpolation="nearest")
ax.set_xticks(range(table.shape[1]), table.columns, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(table.shape[0]), table.index, fontsize=6)
ax.set_ylabel("detector")
fig.colorbar(im, ax=ax, label=METRIC)
plt.show()
table.round(3)

## Per-exposure trends

One value per input exposure: the overscan level for bias, the input level for flats. For bias, inputs flagged as outliers are listed first.

In [ ]:
if trend is not None:
    if "outlier" in trend:
        display(trend[trend["outlier"].fillna(False).astype(bool)]
                .groupby("exposure").agg(n_amps=("amp", "size"), detectors=("detector", "nunique"),
                                         overscan_adu=("overscan_adu", "median")))
    col = next(c for c in ("overscan_adu", "level_adu") if c in trend)
    det = int(df["detector"].min())
    sub = trend[trend["detector"] == det]
    fig, ax = plt.subplots(figsize=(9, 3))
    for amp, g in sub.groupby("amp"):
        ax.plot(g["exposure"], g[col], ".", ms=3, label=f"amp {amp}")
    ax.set(xlabel="exposure", ylabel=col, title=f"detector {det}")
    ax.legend(frameon=False)
    ax.grid(alpha=0.2)
    plt.show()

## Figures written by the run

In [ ]:
figs = sorted((out / "figures").glob("*.png"))
for kind in ("thumb", "fp", "trend", "hist"):
    group = [f for f in figs if f.name.startswith(f"{CAL_TYPE}_{kind}_")]
    if group:
        print(f"--- {kind}: {len(group)} figures")
    for f in group:
        print(f.name)
        display(Image(filename=str(f), width=600))

## Recompute one detector by hand

The metric functions are plain functions of `CalibFrame`s, so a single master can be
re-measured interactively (needs the LSST stack). Set `RUN_LIVE = True` to try it.

In [ ]:
RUN_LIVE = False
if RUN_LIVE and CAL_TYPE == "bias":
    import run_bias_qa
    from common import calio, qa

    source = calio.ButlerSource(os.environ["REPO"], summary["run"]["collection"])
    info = source.list_calibs("bias", detectors=[10])[0]
    rows, live_trend, _ = run_bias_qa.measure_bias(
        source, info, run_bias_qa.bias_params(qa.load_config()))
    display(pd.DataFrame(rows)[["amp", "metric", "value", "unit", "error"]])